In [0]:
%sql
CREATE TABLE IF NOT EXISTS real_state_project.raw_data_real_state.ingestion_map (
    map_id BIGINT GENERATED ALWAYS AS IDENTITY, -- Delta Lake exige BIGINT para IDENTITY
    scraper_tool STRING,                     
    portal STRING,                           
    property_type STRING,                    
    operation_type STRING,                   
    file_prefix STRING,                      
    is_active BOOLEAN                        
);

In [0]:
%sql
-- Limpiamos inserciones de prueba anteriores si es necesario
TRUNCATE TABLE real_state_project.raw_data_real_state.ingestion_map;

-- Insertamos el catálogo maestro de archivos válidos
INSERT INTO real_state_project.raw_data_real_state.ingestion_map
(scraper_tool, portal, property_type, operation_type, file_prefix, is_active)
VALUES
-- Bloque Casas
('Drision', 'inmuebles24', 'casa',               'venta',             'Drision_inmuebles24_casa_venta_',             true),
('Drision', 'inmuebles24', 'casa',               'renta',             'Drision_inmuebles24_casa_renta_',             true),
('Drision', 'inmuebles24', 'casa',               'remates',           'Drision_inmuebles24_casa_remates_',           true),
('Drision', 'inmuebles24', 'casa',               'temporal_vacacional','Drision_inmuebles24_casa_temporal_vacacional_', true),
('Drision', 'inmuebles24', 'casa',               'desarrollos',       'Drision_inmuebles24_casa_desarrollos_',       true),
('Drision', 'inmuebles24', 'casa',               'traspaso',          'Drision_inmuebles24_casa_traspaso_',          true),

-- Bloque Departamentos
('Drision', 'inmuebles24', 'departamento',       'venta',             'Drision_inmuebles24_departamento_venta_',     true),
('Drision', 'inmuebles24', 'departamento',       'renta',             'Drision_inmuebles24_departamento_renta_',     true),

-- Bloque Terrenos e Industriales
('Drision', 'inmuebles24', 'terreno',            'venta',             'Drision_inmuebles24_terreno_venta_',          true),
('Drision', 'inmuebles24', 'industrial',         'venta',             'Drision_inmuebles24_Industrial_venta_',       true),
('Drision', 'inmuebles24', 'industrial',         'renta',             'Drision_inmuebles24_industrial_renta_',       true),

-- Casos Especiales (Mapeos Inteligentes)
('Drision', 'inmuebles24', 'inmueble_general',   'remates',           'Drision_inmuebles24_remates_',                true),
('Drision', 'inmuebles24', 'casa_en_condominio', 'venta_renta',       'Drision_inmuebles24_casa_en_condominio_',     true);

In [0]:
%sql
-- 1. Clonar todo el catálogo de Drision para "Selenium"
INSERT INTO real_state_project.raw_data_real_state.ingestion_map
(scraper_tool, portal, property_type, operation_type, file_prefix, is_active)
SELECT 
    'Selenium' AS scraper_tool,
    portal,
    property_type,
    operation_type,
    REPLACE(file_prefix, 'Drision', 'Selenium') AS file_prefix,
    is_active
FROM real_state_project.raw_data_real_state.ingestion_map
WHERE scraper_tool = 'Drision';

-- 2. Clonar todo el catálogo de Drision para "Selenium1"
INSERT INTO real_state_project.raw_data_real_state.ingestion_map
(scraper_tool, portal, property_type, operation_type, file_prefix, is_active)
SELECT 
    'Selenium1' AS scraper_tool,
    portal,
    property_type,
    operation_type,
    REPLACE(file_prefix, 'Drision', 'Selenium1') AS file_prefix,
    is_active
FROM real_state_project.raw_data_real_state.ingestion_map
WHERE scraper_tool = 'Drision';

In [0]:
%sql
SELECT * FROM real_state_project.raw_data_real_state.ingestion_map;

In [0]:
%sql
ALTER TABLE real_state_project.raw_data_real_state.ingestion_map 
ADD COLUMNS (
    target_table STRING,
    read_options STRING
);

In [0]:
%sql
UPDATE real_state_project.raw_data_real_state.ingestion_map
SET 
    target_table = 'real_state_project.bronze.inmuebles',
    read_options = '{"header": "true", "inferSchema": "true", "delimiter": ","}'
WHERE is_active = true;

In [0]:
%sql
SELECT map_id, scraper_tool, property_type, operation_type, target_table, read_options 
FROM real_state_project.raw_data_real_state.ingestion_map
ORDER BY map_id ASC;